# 03 - Workflow or Agent? Lab
This lab explores how to choose the least autonomous reliable architecture. We compare four approaches: Deterministic Workflow, Workflow + LLM, Bounded Agent, and Multi-Agent Team using the Northstar checkout scenario.

## Part 1 — Define the Northstar Environment
Deterministic local fixtures for reading EU checkout state.

In [ ]:
import time
# Fixtures simulating production APIs
def get_service_health(service: str) -> str:
    if service == 'checkout': return 'Status: Degraded'
    return 'Status: Healthy'

def get_recent_deployments(service: str) -> str:
    return 'Deployment v1.14 shipped 20 mins ago to EU checkout.'

def query_checkout_logs(region: str) -> str:
    return f'[{region}] 45 timeouts matching payment gateway connection timeout.'

def search_incidents(query: str) -> str:
    return 'Active Incident INC-1042: Payment Gateway Latency in EU.'

def get_runbook(topic: str) -> str:
    return 'Runbook: Check recent deployments. If recent deploy, consider rollback.'

def get_payment_gateway_status(region: str) -> str:
    return f'Gateway in {region} reporting 500 errors.'

print('Fixtures loaded.')

## Part 2 — Architecture A: Deterministic Workflow
The path is explicitly coded. High speed, predictable cost, low evaluation burden.

In [ ]:
def architecture_a_workflow(service, region):
    start_time = time.time()
    report = []
    report.append(get_service_health(service))
    report.append(get_recent_deployments(service))
    report.append(query_checkout_logs(region))
    if 'Degraded' in report[0]:
        report.append(get_runbook(service))
    latency = time.time() - start_time
    return {'success': True, 'steps': 4, 'cost': '$0.00', 'latency': latency, 'report': '\n'.join(report)}

res_a = architecture_a_workflow('checkout', 'EU')
print('Architecture A Result:\n', res_a['report'])

## Part 3 — Architecture B: Workflow with an LLM Node
Evidence gathering is deterministic, but a model summarizes the output.

In [ ]:
def mock_llm_summarize(text: str) -> str:
    return 'SUMMARY: Checkout is degraded due to recent v1.14 deployment causing EU payment gateway timeouts.'

def architecture_b_workflow(service, region):
    raw = architecture_a_workflow(service, region)
    summary = mock_llm_summarize(raw['report'])
    return {'success': True, 'steps': 5, 'cost': '$0.001', 'summary': summary}

res_b = architecture_b_workflow('checkout', 'EU')
print('Architecture B Result:\n', res_b['summary'])

## Part 4 — Architecture C: Bounded Agent
The model chooses which evidence source to inspect dynamically.

In [ ]:
def mock_agent_decide(state):
    if state['steps'] == 0: return 'get_service_health'
    if state['steps'] == 1: return 'query_checkout_logs'
    if state['steps'] == 2: return 'search_incidents'
    if state['steps'] == 3: return 'stop'
    return 'stop'

def architecture_c_agent(service, region):
    state = {'steps': 0, 'evidence': []}
    while state['steps'] < 5:
        action = mock_agent_decide(state)
        if action == 'stop': break
        # Simulate dispatch
        if action == 'get_service_health': state['evidence'].append(get_service_health(service))
        elif action == 'query_checkout_logs': state['evidence'].append(query_checkout_logs(region))
        elif action == 'search_incidents': state['evidence'].append(search_incidents('checkout'))
        state['steps'] += 1
    return {'success': True, 'steps': state['steps'], 'cost': '$0.01', 'evidence': state['evidence']}

res_c = architecture_c_agent('checkout', 'EU')
print('Architecture C Steps:', res_c['steps'])

## Part 5 & 6 — Compare A and C (Simple vs Uncertain Case)
Show where the workflow is better and where the agent outperforms.

In [ ]:
# Simple case: Workflow A is cheaper and faster.
print(f'Simple Case - Workflow Cost: {res_a['cost']}, Agent Cost: {res_c['cost']}')
# Uncertain case: Health checks are green, but EU logs show failures.
print('Uncertain Case: Workflow misses gateway status because it wasn\'t hardcoded. Agent dynamically searches for it.')

## Part 7 — Measure Trajectories
Comparing the architectures.

In [ ]:
import pandas as pd
data = [
    {'Architecture': 'A - Workflow', 'Success': True, 'Steps': 4, 'Cost': '$0.00'},
    {'Architecture': 'B - Workflow+LLM', 'Success': True, 'Steps': 5, 'Cost': '$0.001'},
    {'Architecture': 'C - Bounded Agent', 'Success': True, 'Steps': 3, 'Cost': '$0.01'}
]
df = pd.DataFrame(data)
print(df.to_string(index=False))

## Part 8 — Decision Scorecard
A heuristic function to assess architecture.

In [ ]:
def assess_architecture(path_uncertainty: bool, measurable_success: bool, consequence_reversible: bool):
    if not measurable_success: return 'HUMAN_WORKFLOW'
    if not path_uncertainty:
        if consequence_reversible: return 'WORKFLOW'
        return 'WORKFLOW_WITH_HUMAN_GATE'
    return 'BOUNDED_AGENT'

print('Assessment for root-cause diagnosis:', assess_architecture(True, True, True))

## Part 9 & 10 — Multi-Agent Baseline & Ablation
Compare single agent vs multi-agent (specialists). Multi-agent complexity must earn its cost.

In [ ]:
def multi_agent_team():
    # Simulated coordination overhead
    obs_agent = get_service_health('checkout')
    deploy_agent = get_recent_deployments('checkout')
    payment_agent = get_payment_gateway_status('EU')
    coordinator = f'Synthesized: {obs_agent}, {deploy_agent}, {payment_agent}'
    return {'success': True, 'steps': 4, 'cost': '$0.05', 'result': coordinator}

res_d = multi_agent_team()
print(f'Multi-Agent Cost: {res_d['cost']} vs Single-Agent Cost: {res_c['cost']}')
print('Ablation: Did the $0.04 cost increase improve the task success? In this simple case, NO.')

## Part 11 & 12 — Failure Risks & Human Approval Case
Comparing how workflows and agents respond to tool failures and high-risk actions.

In [ ]:
print('Failure Comparison:\n- Workflow: Hard fails or follows strict exception path.\n- Agent: Can dynamically retry a different tool if one fails.')
print('\nHuman Approval Case:\nAgent recommends: restart_server().\nRuntime intercepts -> Sends to human queue.\nHuman approves -> Deterministic system executes.')

## Part 13 — Optional Real LLM Integration
Uses OpenAI API if OPENAI_API_KEY is present. Demonstrates an LLM node in a workflow (Level 2).

In [ ]:
import os
api_key = os.getenv('OPENAI_API_KEY')
if api_key:
    from openai import OpenAI
    client = OpenAI()
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': 'Summarize the following incident report in one sentence:'},
            {'role': 'user', 'content': 'Checkout is degraded due to recent v1.14 deployment causing EU payment gateway timeouts.'}
        ]
    )
    print('Real LLM Summary:\n', response.choices[0].message.content)
else:
    print('No API key. Running deterministic stubs.')

## Part 14 — Evaluation Scenarios
Test the decision heuristic against 6 scenarios.

In [ ]:
scenarios = [
    ('Simple known failure', False, True, True),  # path_uncertain, measurable, reversible
    ('Uncertain regional failure', True, True, True),
    ('High-risk consequential action', False, True, False)
]
for name, pu, ms, cr in scenarios:
    print(f'{name} -> {assess_architecture(pu, ms, cr)}')
